# Multiclass model_metrics walkthrough

Exercises the multiclass additions on a synthetic, imbalanced 5-class problem:

- `summarize_model_performance`: averaged, `per_class`, `split_by_model`
- `show_confusion_matrix`: raw, `normalize`, `cell_values`, `highlight_diagonal`, `return_df`
- `show_roc_curve` and `show_pr_curve`: one-vs-rest per class, macro-average overlay
- `show_calibration_curve`: one-vs-rest per class
- `class_labels` with a model fit on encoded targets (XGBoost)
- binary-only functions rejecting multiclass input

Data is generated in memory, so nothing is written to disk.

## Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

from model_metrics.model_evaluator import (
    summarize_model_performance,
    show_confusion_matrix,
    show_roc_curve,
    show_pr_curve,
    show_calibration_curve,
    show_lift_chart,
    plot_threshold_metrics,
    overlap_summary,
)

## Data

Five imbalanced classes with string labels, so the models see the same kind of target as a real lab or stage outcome.

In [ ]:
class_names = np.array(["Alpha", "Bravo", "Charlie", "Delta", "Echo"])

X, y_idx = make_classification(
    n_samples=3000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_classes=5,
    n_clusters_per_class=1,
    weights=[0.40, 0.25, 0.15, 0.12, 0.08],
    class_sep=1.0,
    random_state=222,
)
X = pd.DataFrame(X, columns=[f"feat_{i}" for i in range(X.shape[1])])
y = pd.Series(class_names[y_idx], name="label")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=222
)
y_test.value_counts().sort_index()

## Models

Logistic Regression and Random Forest train on the string labels. XGBoost requires encoded targets (0 to K-1), so it trains on `LabelEncoder` codes, the same setup that produces the label-mapping warning unless `class_labels` is passed.

In [ ]:
encoder = LabelEncoder().fit(y_train)
lab_names = list(encoder.classes_)  # position = encoded code
print(dict(enumerate(lab_names)))

lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=222).fit(X_train, y_train)
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=222).fit(
    X_train, encoder.transform(y_train)
)

models = [lr, rf, xgb]
model_titles = ["Logistic Regression", "Random Forest", "XGBoost"]
print("XGBoost classes_:", xgb.classes_)

## summarize_model_performance

### Averaged metrics (macro)

In [ ]:
summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    average="macro",
    return_df=True,
)

### Weighted average

In [ ]:
summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    average="weighted",
    return_df=True,
)

### Per class, one MultiIndex table

In [ ]:
summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    per_class=True,
    return_df=True,
)

### Per class, one DataFrame per model

In [ ]:
per_class_dfs = summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    per_class=True,
    split_by_model=True,
    return_df=True,
)
per_class_dfs["XGBoost"]

### From precomputed probabilities

In [ ]:
y_probs = [m.predict_proba(X_test) for m in models]

summarize_model_performance(
    y_prob=y_probs,
    y=y_test,
    model_title=model_titles,
    class_labels=lab_names,
    return_df=True,
)

## show_confusion_matrix

### Raw counts

In [ ]:
show_confusion_matrix(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    subplots=True,
    n_cols=3,
    figsize=(6, 6),
    class_report=True,
)

### Row-normalized (diagonal = recall), proportion and count, diagonal bordered

In [ ]:
cm_dfs = show_confusion_matrix(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    subplots=True,
    n_cols=3,
    figsize=(6, 6),
    normalize="true",
    highlight_diagonal=True,
    return_df=True,
)
cm_dfs["Random Forest"].round(3)

### Column-normalized (diagonal = precision), proportions only

In [ ]:
show_confusion_matrix(
    model=xgb,
    model_title="XGBoost",
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    figsize=(6, 6),
    normalize="pred",
    cell_values="proportion",
    highlight_diagonal=True,
    diagonal_kwgs={"edgecolor": "red", "linewidth": 1.5},
)

## show_roc_curve

### One-vs-rest curve per class

In [ ]:
show_roc_curve(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    subplots=True,
    n_cols=3,
    figsize=(20, 6),
    decimal_places=3,
)

### Macro-average curve per model

In [ ]:
show_roc_curve(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    overlay=True,
    decimal_places=3,
)

## show_pr_curve

In [ ]:
show_pr_curve(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    subplots=True,
    n_cols=3,
    figsize=(20, 6),
    legend_metric="ap",
    decimal_places=3,
)

In [ ]:
show_pr_curve(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    overlay=True,
    legend_metric="aucpr",
    decimal_places=3,
)

## show_calibration_curve

In [ ]:
show_calibration_curve(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    subplots=True,
    n_cols=3,
    figsize=(20, 6),
    bins=8,
)

## Label mapping without `class_labels`

XGBoost's `classes_` are 0 to 4 while `y_test` holds names. Without `class_labels`, the function falls back to the sorted names and warns once. The result matches the `class_labels` run because `LabelEncoder` sorts.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    no_labels = summarize_model_performance(
        model=xgb, X=X_test, y=y_test, per_class=True, return_df=True
    )
print(caught[0].message if caught else "no warning")

with_labels = summarize_model_performance(
    model=xgb, X=X_test, y=y_test, class_labels=lab_names, per_class=True, return_df=True
)
print("Identical:", no_labels.equals(with_labels))

## Sanity checks against scikit-learn

In [ ]:
macro = summarize_model_performance(
    model=models,
    model_title=model_titles,
    X=X_test,
    y=y_test,
    class_labels=lab_names,
    average="macro",
    return_df=True,
)
recall_row = macro.loc[macro["Metrics"] == "Sensitivity/Recall"]
auc_row = macro.loc[macro["Metrics"] == "AUC ROC"]

y_test_codes = encoder.transform(y_test)
checks = []
for title, p in zip(model_titles, y_probs):
    pred_codes = p.argmax(axis=1)
    checks.append(
        {
            "Model": title,
            "Macro AUC (sklearn)": round(roc_auc_score(y_test_codes, p, multi_class="ovr"), 3),
            "Macro AUC (model_metrics)": auc_row[title].iloc[0],
            "Balanced accuracy (sklearn)": round(balanced_accuracy_score(y_test_codes, pred_codes), 3),
            "Macro recall (model_metrics)": recall_row[title].iloc[0],
        }
    )
pd.DataFrame(checks)

Macro recall equals balanced accuracy by definition, so the last two columns should agree, as should the two AUC columns.

## Binary-only functions reject multiclass input

In [ ]:
for name, call in [
    ("show_lift_chart", lambda: show_lift_chart(model=rf, X=X_test, y=y_test)),
    ("plot_threshold_metrics", lambda: plot_threshold_metrics(model=rf, X_test=X_test, y_test=y_test)),
    ("overlap_summary", lambda: overlap_summary(y_test, y_pred_a=rf.predict(X_test), y_pred_b=lr.predict(X_test))),
    ("show_roc_curve(delong=...)", lambda: show_roc_curve(y_prob=y_probs[0], y=y_test, delong=(y_probs[0], y_probs[1]))),
    ("show_confusion_matrix(custom_threshold=...)", lambda: show_confusion_matrix(model=rf, X=X_test, y=y_test, custom_threshold=0.3)),
]:
    try:
        call()
        print(f"{name}: no error (unexpected)")
    except ValueError as e:
        print(f"{name}: {e}")